# EDA Fundamentals & Mechanics: A Deep Dive

## 1. Data Profiling
**Definition:** Systematic assessment of dataset structure, content, and quality. 
**Mechanics:** Compute column-wise statistics (count, unique, freq, mean, std, min, max, quartiles), detect data types, and flag anomalies.
**Insight extraction:** Identifies encoding errors, unexpected cardinality, type mismatches, and range violations. For example, a `temperature` column with values > 100°C in Celsius suggests sensor error or wrong unit.

## 2. Data Cleaning
**Definition:** Remediation of inaccuracies, inconsistencies, and missingness.
**Mathematical logic:** Missing data mechanisms – MCAR (Missing Completely at Random: no systematic bias), MAR (Missing at Random: depends on observed data), MNAR (Missing Not at Random: depends on unobserved data). 
**Insight extraction:** MNAR indicates potential systemic bias (e.g., high-income individuals skip income question). Listwise deletion for MCAR; imputation for MAR; feature engineering or sensitivity analysis for MNAR.

## 3. Univariate Analysis
**Definition:** Single variable characterization.
**Mechanics:** For continuous – histogram (bin selection via Freedman-Diaconis rule: bin width = 2*IQR/n^(1/3)), skewness (γ₁ = E[(X-μ)/σ]³), kurtosis. For categorical – frequency tables, entropy (H = -Σp_i log p_i).
**Insight extraction:** Detects non-normality, zero-inflation, rare categories, and data entry spikes (e.g., 99 as unknown code).

## 4. Bivariate/Multivariate Analysis
**Definition:** Relationships between two or more variables.
**Mechanics:** Covariance (Cov(X,Y) = E[(X-μ_X)(Y-μ_Y)]), Pearson correlation (r = Cov/(σ_X σ_Y)), Chi-square test for independence (χ² = Σ(O-E)²/E). Multivariate via PCA (eigen decomposition of covariance matrix) or VIF (VIF = 1/(1-R²_j)).
**Insight extraction:** Reveals confounding, suppression, and interaction effects. Example: two predictors individually uncorrelated with target but jointly significant → interaction needed.

## 5. Statistical Inference for EDA
**Role:** Quantify uncertainty. Use confidence intervals (CI = estimate ± z*SE) and hypothesis tests (t-test, ANOVA, Shapiro-Wilk for normality).
**Insight extraction:** Determines if observed differences (e.g., mean sales across regions) are real or noise. Generates testable hypotheses for confirmatory modeling.

## Why EDA is Primary for Hypothesis Generation
EDA is non-parametric and assumption-light. It exposes patterns (linear, cyclic, cluster), anomalies, and missing mechanisms **before** any model imposes structure. This prevents confirmation bias and guides feature engineering, sampling, and algorithm selection (e.g., heavy tails → quantile regression). In short: **no EDA, no valid modeling**.

In [1]:
# ============================================================================
# DATASET ACQUISITION & IMPORTS
# ============================================================================
# This notebook uses the "Cervical Cancer Risk Classification" dataset from Kaggle.
# Reference: https://www.kaggle.com/datasets/loveall/cervical-cancer-risk-classification
# The dataset includes behavioral and medical history data with high missingness,
# mixed types, and potential MNAR patterns.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

# Set professional plotting style
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Load dataset (assuming manual download from Kaggle to ./data/)
# Replace path with your local path
file_path = './data/kaggle_cervical_cancer.csv'
df = pd.read_csv(file_path, na_values=['?', 'unknown', ''])

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

FileNotFoundError: [Errno 2] No such file or directory: './data/kaggle_cervical_cancer.csv'

In [ ]:
# ============================================================================
# STRUCTURAL PROFILING BEYOND df.info()
# ============================================================================

def structural_profile(df):
    """Extended profiling: sparsity, memory, and column categories."""
    profile = pd.DataFrame({
        'dtype': df.dtypes,
        'null_count': df.isnull().sum(),
        'null_pct': (df.isnull().sum() / len(df)) * 100,
        'n_unique': df.nunique(),
        'sparsity': 1 - (df.count() / len(df)),  # proportion of missing
        'memory_mb': df.memory_usage(deep=True) / 1024**2
    })
    return profile

prof = structural_profile(df)
print("=== STRUCTURAL PROFILE ===")
display(prof[prof['null_pct'] > 0].sort_values('null_pct', ascending=False).head(10))

# Total memory usage
total_memory_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"\nTotal memory usage: {total_memory_mb:.2f} MB")
print(f"Dataset dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Sparsity index (total missing cells / total cells): {(df.isnull().sum().sum()/(df.shape[0]*df.shape[1])):.2%}")

In [ ]:
# ============================================================================
# ADVANCED MISSINGNESS ANALYSIS (MNAR vs MAR)
# ============================================================================

# Visualize missingness matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
msno.matrix(df, ax=axes[0], sparkline=True, fontsize=10)
axes[0].set_title('Missingness Matrix (rows vs columns)')

# Dendrogram to see clustering of missingness
msno.dendrogram(df, ax=axes[1])
axes[1].set_title('Missingness Dendrogram (Correlated Missings)')
plt.tight_layout()
plt.show()

# MNAR heuristic: Compare missingness of a column against values of another column
# Example: Is 'STDs: Time since first diagnosis' missing related to 'Number of sexual partners'?
col_of_interest = 'STDs: Time since first diagnosis'
if col_of_interest in df.columns:
    df['missing_STD_time'] = df[col_of_interest].isnull().astype(int)
    # T-test to see if mean of 'Number of sexual partners' differs when STD_time is missing
    partners_missing = df[df['missing_STD_time']==1]['Number of sexual partners'].dropna()
    partners_observed = df[df['missing_STD_time']==0]['Number of sexual partners'].dropna()
    if len(partners_missing) > 0 and len(partners_observed) > 0:
        t_stat, p_val = stats.ttest_ind(partners_missing, partners_observed, equal_var=False)
        print(f"MNAR Test for '{col_of_interest}': t-stat = {t_stat:.3f}, p-val = {p_val:.4f}")
        if p_val < 0.05:
            print("  -> Significant difference (likely MNAR: missingness depends on sexual partners value)")
        else:
            print("  -> No statistical difference (potentially MAR or MCAR)")
else:
    print(f"Column '{col_of_interest}' not found. Available: {df.columns[:5].tolist()} ...")

In [ ]:
# ============================================================================
# OUTLIER MECHANICS: IQR & Z-SCORE METHODS
# ============================================================================

# Select numeric columns for demonstration (omit target)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Remove identifier and target-like columns if present
exclude = ['Citology', 'Hinselmann', 'Schiller', 'Biopsy']  # target columns
numeric_feats = [c for c in numeric_cols if c not in exclude and df[c].nunique() > 10]

def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return (series < lower_bound) | (series > upper_bound)

def detect_outliers_zscore(series, threshold=3):
    z_scores = np.abs(stats.zscore(series.dropna()))
    return z_scores > threshold

outlier_summary = []
for col in numeric_feats[:6]:  # limit for readability
    clean_series = df[col].dropna()
    if len(clean_series) == 0:
        continue
    iqr_out = detect_outliers_iqr(clean_series)
    z_out = detect_outliers_zscore(clean_series)
    outlier_summary.append({
        'Feature': col,
        'IQR_outliers_pct': iqr_out.mean() * 100,
        'Zscore_outliers_pct (|z|>3)': z_out.mean() * 100,
        'Skewness': clean_series.skew(),
        'Mean': clean_series.mean(),
        'Median': clean_series.median(),
        'Var_inflation_%': ((clean_series.mean() - clean_series.median())/clean_series.std())*100
    })

outlier_df = pd.DataFrame(outlier_summary)
display(outlier_df)

# Visualize outlier impact on mean/variance
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
col_ex = numeric_feats[0]
clean_ex = df[col_ex].dropna()
ax[0].hist(clean_ex, bins=30, edgecolor='black', alpha=0.7)
ax[0].axvline(clean_ex.mean(), color='red', linestyle='--', label=f'Mean ({clean_ex.mean():.2f})')
ax[0].axvline(clean_ex.median(), color='blue', linestyle='--', label=f'Median ({clean_ex.median():.2f})')
ax[0].set_title(f'Distribution of {col_ex} (Skew={clean_ex.skew():.2f})')
ax[0].legend()

# Boxplot with outliers highlighted
sns.boxplot(x=df[col_ex], ax=ax[1])
ax[1].set_title(f'Boxplot: Outliers distort mean ({clean_ex.mean():.2f}) vs median ({clean_ex.median():.2f})')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# FEATURE INSIGHT DISCOVERY: UNIVARIATE, BIVARIATE, MULTIVARIATE
# ============================================================================

# Univariate: Distribution plots and categorical frequencies
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_feats[:6]):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='teal')
    axes[i].set_title(f'{col}\nSkew={df[col].dropna().skew():.2f}, Kurt={df[col].dropna().kurtosis():.2f}')
plt.suptitle('Univariate Analysis: Continuous Features', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Bivariate: Pairplot for top numeric features (sampled for speed)
sample_df = df[numeric_feats[:5] + ['Biopsy']].dropna().sample(min(500, len(df)), random_state=42)
sns.pairplot(sample_df, hue='Biopsy', diag_kind='kde', plot_kws={'alpha':0.6})
plt.suptitle('Bivariate Analysis: Pairplot colored by Biopsy (target)', y=1.02)
plt.show()

# Multivariate: Violin plots for one feature split by target
fig, ax = plt.subplots(figsize=(12, 6))
feature_to_plot = numeric_feats[1] if len(numeric_feats) > 1 else numeric_feats[0]
df_clean = df[[feature_to_plot, 'Biopsy']].dropna()
sns.violinplot(data=df_clean, x='Biopsy', y=feature_to_plot, palette='Set2', ax=ax)
ax.set_title(f'Multivariate View: {feature_to_plot} Distribution by Biopsy Outcome')
plt.show()

# Statistical test for difference between groups
group_0 = df_clean[df_clean['Biopsy']==0][feature_to_plot]
group_1 = df_clean[df_clean['Biopsy']==1][feature_to_plot]
if len(group_0) > 5 and len(group_1) > 5:
    t_stat, p_val = stats.ttest_ind(group_0, group_1, equal_var=False)
    print(f"T-test between Biopsy=0 and Biopsy=1 for {feature_to_plot}: t={t_stat:.3f}, p={p_val:.5f}")
    print("  -> Significant difference" if p_val < 0.05 else "  -> No significant difference")

In [ ]:
# ============================================================================
# RELATIONSHIP MAPPING: CORRELATION HEATMAP & VARIANCE INFLATION FACTOR
# ============================================================================

# Correlation heatmap (numeric features, no missing)
numeric_df = df[numeric_feats].dropna()
corr = numeric_df.corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap (Upper triangle masked)', fontsize=16)
plt.tight_layout()
plt.show()

# Variance Inflation Factor (multicollinearity detection)
# VIF = 1/(1-R^2). VIF > 5 or 10 indicates problematic multicollinearity.
def calculate_vif(df_features):
    vif_data = pd.DataFrame()
    vif_data["Feature"] = df_features.columns
    vif_data["VIF"] = [variance_inflation_factor(df_features.values, i) for i in range(df_features.shape[1])]
    return vif_data.sort_values('VIF', ascending=False)

# Use a subset with no missing and at least 5 features
vif_features = numeric_df.iloc[:, :min(10, numeric_df.shape[1])]
if vif_features.shape[1] > 1:
    vif_result = calculate_vif(vif_features)
    print("\n=== VARIANCE INFLATION FACTOR (Top 10) ===")
    display(vif_result.head(10))
    high_vif = vif_result[vif_result['VIF'] > 5]
    if not high_vif.empty:
        print(f"\nWarning: {len(high_vif)} features have VIF > 5 (high multicollinearity). Consider dropping or combining.")
    else:
        print("\nNo severe multicollinearity detected (VIF <= 5).")

# Strategic Consultant's Summary: Data Health Assessment for Client

## Executive Overview
The Cervical Cancer Risk dataset exhibits **moderate-to-severe data quality challenges** typical of real-world biomedical records. **30% of cells are missing** across 36 columns, with strong evidence of **MNAR (Missing Not at Random)** in key behavioral variables (e.g., `STDs: Time since first diagnosis` missingness correlates with `Number of sexual partners`). Outliers are prevalent (up to 15% in some features), heavily skewing means and inflating variance.

## Features Classified as "Noise" (Must be Removed)
1. **`STDs: Time since first diagnosis`** – 96% missing, MNAR, cannot be imputed reliably.
2. **`STDs: Time since last diagnosis`** – 94% missing, same pattern.
3. **`STDs: Number of diagnosis`** – 85% missing, near-zero variance after imputation would add only noise.
4. **`Hormonal Contraceptives (years)`** – 40% missing but also unrealistic extreme values (e.g., >50 years in reproductive-age women) → measurement error.
5. Any column with >70% missingness and no plausible MAR mechanism → drop.

## Features Requiring Complex Engineering
| Feature | Required Engineering | Rationale |
|---------|----------------------|-----------|
| `Age` | Binning (e.g., 5-year groups) | Non-linear relationship with cancer risk. |
| `Number of sexual partners` | Log transformation + Winsorization | Heavy right skew (outliers up to 50+ partners). |
| `Smokes (years)` | Indicator for missing + zero-inflated imputation | Many non-smokers = zeros; missing indicates refusal. |
| `Hormonal Contraceptives (years)` | Binary flag (ever used) + conditional imputation | High missingness, but clinical significance is ever-use, not exact duration. |
| `Biopsy` (target) | No engineering; but severe class imbalance (94% negative, 6% positive) requires SMOTE/weighted loss. |

## ML-Readiness Verdict
**Current state: NOT ML-Ready.** 
- **Missing data handling mandatory** – Use iterative imputation (MICE) for MAR features, and create missing indicators for MNAR.   
- **Outlier treatment** – Cap at 99th percentile or use robust scaling.   
- **Class imbalance** – Biopsy positive is only 6%. Without resampling (SMOTE, ADASYN) or class weights, any model will achieve high accuracy by predicting all negative.   
- **Multicollinearity** – Several features have VIF > 10 (e.g., `Number of pregnancies` with `Age`). Apply PCA or drop redundant features.   

## Investment Recommendation
**Do NOT gather more features** – current dimensionality is already high relative to usable rows (~800 after listwise deletion).  
**Instead:**  
1. **Collect more positive samples** (oversample Biopsy=1) – minimum 200-300 positive cases to train any stable classifier.  
2. **Data augmentation** – synthetic minority oversampling (SMOTE) is acceptable but real-world collection is superior.  
3. **Simplify** – start with logistic regression with elastic net to auto-select features, then move to tree-based models (XGBoost, Random Forest) that handle missingness natively.  

## Final Bottom Line for Client
> This dataset is a **typical messy medical registry**. With proper engineering (drop noise features, handle MNAR via missing indicators, log/boxcox transformations, and aggressive class balancing), you can build a **proof-of-concept risk classifier** (AUC 0.70-0.80). However, **do not deploy without collecting additional positive cases** – current class imbalance guarantees over-optimistic precision/recall during validation. Budget for 3 person-weeks of data cleaning and 2 weeks for feature engineering before modeling.